# M6a 결정트리 — 실습 (W9, M6 2부작 1편)

> 위에서부터 한 셀씩 `Shift+Enter`로 실행하세요. `___` 빈칸은 직접 채웁니다.

**이 실습이 끝나면**
1. 미니 승객 10명의 **지니 손계산**(0.5 / 0.32 / 0.375, 이득 0.18 vs 0.125)을 코드로 재현하고 **sklearn 검산**으로 일치를 확인한다 ⭐
2. 와인 트리를 `plot_tree`로 **읽고**, 루트 지니 0.6573을 3클래스 손계산으로 검증한다
3. 깊이 스윕으로 **과적합**(M2a의 재회)을, 스케일링 실험으로 **트리 vs KNN**을, 부트스트랩으로 **불안정성**을 관찰한다

**7단계 멘탈모델 초점:** 모델(Model)

## Part A. 세트피스 — 세상에서 가장 작은 결정트리 ⭐
미니 승객 10명: 생존 5(여 4·남 1) / 사망 5(여 1·남 4), 아이 2명(전원 생존). 두 후보 질문 — **"여성인가?" vs "아이인가?"** — 를 지니 불순도로 대결시킵니다. 먼저 종이에서 계산해 보고, 코드로 검산하세요.

In [ ]:
import numpy as np                                     # 수치 계산

def gini(counts):                                      # counts = [사망 수, 생존 수]
    n = sum(counts)                                    # 그룹 인원
    p0, p1 = counts[0] / n, counts[1] / n              # 두 클래스의 비율
    return 1 - (p0**2 + ___**2)                        # ✍️ 빈칸: 지니 = 1 - Σp² (나머지 비율의 제곱은?)

print('루트(사망5·생존5):', round(gini([5, 5]), 4))      # 0.5 — 반반은 2클래스 최악
g_f = gini([1, 4])                                     # 여성 5명: 사망1·생존4
g_m = gini([4, 1])                                     # 남성 5명: 사망4·생존1
g_child = gini([0, 2])                                 # 아이 2명: 전원 생존
g_adult = gini([5, 3])                                 # 어른 8명: 사망5·생존3
print('여성:', round(g_f, 4), '| 남성:', round(g_m, 4))          # 0.32 / 0.32
print('아이:', round(g_child, 4), '| 어른:', round(g_adult, 5))  # 0.0 / 0.46875 — 완전 순수 잎 등장!

In [ ]:
w_sex = 0.5 * g_f + 0.5 * ___                          # ✍️ 빈칸: 성별 질문의 가중평균(5/10씩 — 남성 쪽 지니는?)
w_child = 0.2 * g_child + 0.8 * g_adult                # 아이 질문의 가중평균(2/10, 8/10)
print('가중평균 — 성별:', round(w_sex, 4), '| 아이:', round(w_child, 4))        # 0.32 / 0.375
print('이득(불순도 감소) — 성별:', round(0.5 - w_sex, 4), '| 아이:', round(0.5 - w_child, 4))  # 0.18 / 0.125
print('트리의 선택:', '여성인가?' if 0.5 - w_sex > 0.5 - w_child else '아이인가?')  # 이득 큰 쪽!

In [ ]:
from sklearn.tree import DecisionTreeClassifier        # 결정트리

Xm = np.array([[1, 0], [1, 0], [1, 0], [1, 1], [1, 0],  # 여성 5명 — 열: [성별(여=1), 아이(1)]
               [0, 1], [0, 0], [0, 0], [0, 0], [0, 0]])  # 남성 5명
ym = np.array([1, 1, 1, 1, 0, 1, 0, 0, 0, 0])           # 생존=1 · 사망=0
mini = DecisionTreeClassifier(max_depth=1, random_state=0).fit(Xm, ym)  # 질문 1개짜리 트리
print('루트가 고른 특징 index:', mini.tree_.feature[0])   # 0(성별)이면 손계산 판정과 일치
print('노드별 지니:', np.round(mini.tree_.impurity, 4))   # [0.5, 0.32, 0.32] — 손계산 그대로?

> **검산 포인트:** 루트 특징 = **성별**(이득 0.18 > 0.125), 노드 지니 = **[0.5, 0.32, 0.32]** — 손계산과 일치. sklearn은 같은 산수를 **모든 특징 × 모든 임계값**에 반복할 뿐입니다. 그리고 "아이인가?"는 지니 **0짜리 완전 순수한 잎**을 만들고도 졌습니다 — 그 잎엔 겨우 2명. **가중평균은 작은 완벽에 속지 않습니다.**

## Part B. 와인 트리 읽기 — 화이트박스
와인 178병(3품종)을 70:30으로 나누고 트리를 학습해, **모든 질문을 사람이 읽어** 봅니다. 루트 노드의 지니는 3클래스 손계산으로 검증합니다(0.5를 넘을 수 있어요!).

In [ ]:
import matplotlib.pyplot as plt                        # 그래프
from sklearn.datasets import load_wine                 # 와인 데이터(3품종)
from sklearn.model_selection import train_test_split   # 공정한 시험(M2a)
from sklearn.tree import plot_tree                     # 트리 시각화

wine = load_wine()                                     # 178병, 특징 13개
X, y = wine.data, wine.target                          # X=성분, y=품종(0/1/2)
X_train, X_test, y_train, y_test = train_test_split(   # 30% 시험, 비율 유지
    X, y, test_size=0.3, random_state=42, stratify=y)

tree = DecisionTreeClassifier(max_depth=___, random_state=0)  # ✍️ 빈칸: 읽기 좋은 깊이(깊이 스윕의 최고 깊이와 동일)
tree.fit(X_train, y_train)                             # 학습
print('train:', round(tree.score(X_train, y_train), 3),
      '| test:', round(tree.score(X_test, y_test), 3)) # 0.992 / 0.981

inner = tree.tree_                                     # 학습된 트리 내부 구조
print('루트 질문:', wine.feature_names[inner.feature[0]], '<=', round(inner.threshold[0], 2))  # 첫 갈림길
print('루트 지니(sklearn):', round(inner.impurity[0], 4))   # 0.6573
counts = np.bincount(y_train)                          # train 품종 분포 [41 50 33]
hand = 1 - sum((c / len(y_train))**2 for c in counts)  # 3클래스 지니 손계산
print('루트 지니(손계산) :', round(hand, 4), '| train 분포:', counts)  # 일치?

plt.figure(figsize=(16, 8))                            # 큰 그림 영역
plot_tree(tree, feature_names=wine.feature_names,      # 노드에 특징 이름
          class_names=list(wine.target_names),         # 잎에 품종 이름
          filled=True, fontsize=8)                     # 색칠
plt.show()                                             # 질문을 문장으로 읽어 보자

> **읽기 포인트:** 루트 = `color_intensity <= 3.82?`, 지니 **0.6573** — train 분포 [41, 50, 33]의 손계산과 일치. 2클래스 최악은 0.5지만 **3클래스 최악은 ≈0.667**(균등 1/3). 각 노드의 `gini`=불순도, `samples`=도달 샘플 수, `value`=클래스별 개수 — 위에서부터 따라 읽으면 예측 규칙이 그대로 문장이 됩니다(화이트박스).

## Part C. 깊이 ↔ 과적합 — M2a 깊이 스윕의 재회
3주 차(M2a)에 유방암 데이터로 했던 실험을 와인에서 반복합니다 — 이번엔 **상자를 연 채로**: 깊이를 풀면 잎마다 지니 0이 될 때까지 나눈다 = train 암기.

In [ ]:
depths = range(1, 16)                                  # 깊이 1~15
train_acc, test_acc = [], []                           # 두 점수 저장
for d in depths:                                       # 복잡도를 키워가며
    t = DecisionTreeClassifier(max_depth=d, random_state=0).fit(X_train, y_train)  # 깊이 d 트리
    train_acc.append(t.score(X_train, y_train))        # 학습 점수
    test_acc.append(t.score(X_test, ___))              # ✍️ 빈칸: 시험 점수(시험셋의 정답은?)

plt.plot(depths, train_acc, 'o-', label='train')       # 두 곡선을 나란히(M2a 진단법)
plt.plot(depths, test_acc, 's-', label='test')
plt.xlabel('max_depth (model complexity)'); plt.ylabel('accuracy')  # 축(영어)
plt.legend(); plt.grid(True, alpha=0.3)
plt.title('Wine: tree depth vs accuracy')
plt.show()
print('depth 1 :', round(train_acc[0], 3), '/', round(test_acc[0], 3))    # 0.661 / 0.611 과소적합
print('depth 3 :', round(train_acc[2], 3), '/', round(test_acc[2], 3))    # 0.992 / 0.981 최고
print('depth 15:', round(train_acc[-1], 3), '/', round(test_acc[-1], 3))  # 1.000 / 0.963 암기

> **관찰:** 깊이 1(0.611)은 질문 하나로 부족(과소적합), 최고 test는 **깊이 3(0.981)**, 깊이 4부터 train **만점** — test는 0.963으로 하락. **train 만점 = 잎마다 지니 0까지 나눴다 = 통째로 외웠다**(M2a의 경고, 이제 이유까지). 참고: train이 1.0이 된 뒤로는 나눌 불순물이 없어 트리가 더 자라지 않습니다(4~15 동일). 적절한 깊이는 test가 아니라 **교차검증**으로(M2b — test로 고르면 누수!).

## Part D. 트리는 스케일링이 필요 없다 — M3 KNN과 정반대
질문 `x <= 3.82?`는 값의 **대소(순서)** 만 봅니다. 첫 특징만 ×100 해서 트리와 KNN의 반응을 비교해 보세요.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier     # 비교용 KNN(M3 — 거리 기반)

Xtr_s, Xte_s = X_train.copy(), X_test.copy()           # 복사본에서 실험
Xtr_s[:, 0] *= ___                                     # ✍️ 빈칸: 첫 특징만 100배로(몇을 곱하나?)
Xte_s[:, 0] *= 100                                     # 시험셋도 같은 변환

t1 = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X_train, y_train).score(X_test, y_test)  # 트리-원본
t2 = DecisionTreeClassifier(max_depth=3, random_state=0).fit(Xtr_s, y_train).score(Xte_s, y_test)     # 트리-x100
k1 = KNeighborsClassifier(5).fit(X_train, y_train).score(X_test, y_test)   # KNN-원본
k2 = KNeighborsClassifier(5).fit(Xtr_s, y_train).score(Xte_s, y_test)      # KNN-x100
print('트리: 원본', round(t1, 3), '→ x100', round(t2, 3))  # 0.981 → 0.981 미동 없음
print('KNN : 원본', round(k1, 3), '→ x100', round(k2, 3))  # 0.722 → 0.796 출렁!

> **관찰:** 트리는 **0.981 → 0.981 미동 없음** — 특징을 ×100 하면 임계값도 함께 ×100(`x <= 382?`)이 될 뿐, 나뉘는 샘플이 같습니다. KNN은 0.722 → 0.796으로 **출렁** — 이번엔 우연히 올랐지만, **스케일에 반응한다는 것 자체가 문제**(M3의 교훈: KNN은 스케일링 필수). 와인에서 KNN이 0.722에 그치는 이유도 스케일링을 안 했기 때문 — 트리는 그 걱정이 없습니다.

## Part E. 불안정성 — 첫 질문부터 바뀐다 (M6b 예고)
같은 train에서 **복원추출(bootstrap)** 로 살짝 다른 표본 두 벌을 만들어, 각각 깊이 2 트리를 키워 봅니다. 첫 질문(루트)이 같을까요?

In [ ]:
n = len(X_train)                                       # train 크기(124)
fig, axes = plt.subplots(1, 2, figsize=(15, 5))        # 두 트리를 나란히
for i, seed in enumerate([0, 1]):                      # 표본 두 벌
    rng = np.random.default_rng(seed)                  # 재현성
    idx = rng.integers(0, n, ___)                      # ✍️ 빈칸: 복원추출 — 원본과 같은 크기만큼 뽑기(변수 하나)
    tb = DecisionTreeClassifier(max_depth=2, random_state=0).fit(X_train[idx], y_train[idx])  # 표본으로 학습
    print('표본', i + 1, '의 루트:', wine.feature_names[tb.tree_.feature[0]],
          '<=', round(tb.tree_.threshold[0], 3))       # 첫 질문 비교
    plot_tree(tb, feature_names=wine.feature_names, class_names=list(wine.target_names),
              filled=True, fontsize=8, ax=axes[i])     # 시각화
    axes[i].set_title('Bootstrap sample ' + str(i + 1))  # 제목(영어)
plt.tight_layout(); plt.show()

> **관찰:** 표본 1의 루트는 `flavanoids <= 1.335`, 표본 2는 `color_intensity <= 3.945` — **표본이 살짝 달라지자 첫 질문부터 다릅니다.** 상위권 질문들의 이득이 비슷하면 1등이 쉽게 뒤바뀌고, 루트가 바뀌면 아래 전체가 연쇄로 바뀝니다(고분산·불안정). **다음 주(M6b):** 이 흔들림을 **투표**로 상쇄시켜 숲을 짓습니다 — 정확도 70% 나무 3그루의 다수결은 몇 %가 될까요? 미리 생각해 보세요.

## 🤖 AI 코파일럿 활용 (선택) — ai-native v1
막히면 AI 튜터에게 묻되, **먼저 스스로 생각**하고 답을 **실행으로 검증**하세요.

**좋은 질문 예시**
- "6:4 그룹의 지니를 내가 계산해 볼 테니 채점해 줘."
- "'아이인가?'가 지니 0짜리 잎을 만들고도 진 이유를 가중평균으로 설명해 볼게 — 허점을 찔러 줘."
- "트리가 스케일링이 필요 없는 이유를 KNN과 대비해 설명해 볼게."
- "min_samples_leaf를 키우면 트리가 어떻게 변할지 예측해 볼게 — 실행으로 검증할게."

**가드레일**
1. 먼저 손으로 생각 → 그 다음 AI
2. AI 코드는 *왜 그런지* 설명할 수 있을 때만 사용
3. AI 출력은 실행으로 검증

## 정리 & 자가 점검

**오늘 한 일 3줄**
1. 미니 승객 10명의 지니 손계산(이득 0.18 vs 0.125)을 재현하고 **sklearn 검산 일치**([0.5, 0.32, 0.32], 루트=성별)를 확인했다
2. 와인 트리를 읽고(루트 `color_intensity <= 3.82`, 지니 0.6573 손계산 일치), 깊이 스윕으로 과적합(최고 test=깊이 3, 0.981)을 재회했다
3. 스케일링 실험(트리 미동 없음 vs KNN 출렁)과 부트스트랩 불안정성(루트부터 다른 두 트리)을 관찰했다

**스스로 점검**
- [ ] 지니 = 1 − Σp²를 쓰고, 8:2 그룹의 지니(0.32)를 암산할 수 있다
- [ ] "작은 완벽(지니 0 잎)이 지는 이유"를 가중평균으로 설명할 수 있다
- [ ] 트리가 스케일링이 필요 없는 이유를 한 문장으로 말할 수 있다
- [ ] train 만점 트리를 보면 무엇을 의심해야 하는지 안다(M2a)
- [ ] 불안정성이 왜 생기는지, 왜 다음 주 숲의 재료가 되는지 말할 수 있다

**🔹심화 (선택)**
- `criterion='entropy'`로 바꿔 Part B를 재실행 — **루트 질문이 바뀌나요?** (이 데이터·분할에선 flavanoids로 바뀝니다 — 두 자가 항상 같은 트리를 만들지는 않습니다)
- `min_samples_leaf=5`를 주고 깊이 제한 없이 키워 보세요 — 다른 손잡이로도 과적합이 제어되나요?
- **Part E의 시드를 `[0, 1]`에서 `range(12)`로 늘려** 루트에 어떤 특징이 몇 번 등장하는지 세어 보세요 — 몇 개의 특징이 경합하나요? (답은 정답 파일에 — 다음 주에 이 명단과 재회합니다)
- `export_text(tree, feature_names=...)`로 트리를 **텍스트 규칙**으로 뽑아 읽어 보세요.

**다음 시간(M6b):** 다수결의 수학 · 배깅 · 특징 중요도 — 흔들리는 나무들을 숲으로.